# Phase 9: Encoder/Text Features

This notebook creates text-based features for Stage 2 fraud detection using a pretrained encoder model.

**What we're doing:**
- Using `all-MiniLM-L6-v2` (a lightweight sentence transformer)
- Computing semantic similarity between application fields and text evidence
- Creating keyword-based risk signals

**What we're NOT doing:**
- Training or fine-tuning any models
- Using heavy deep learning
- Requiring GPU

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Add src to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from encoder_features import (
    load_borderline_dataset,
    load_cleaned_dataset,
    merge_identity_fields,
    load_encoder_model,
    build_application_identity_text,
    build_address_reference_text,
    build_employer_reference_text,
    create_similarity_features,
    create_text_length_features,
    create_keyword_features,
    create_text_feature_table,
    save_outputs,
    summarize_features_by_label,
    SUSPICIOUS_KEYWORDS,
    ENCODER_MODEL_NAME,
)

print("Imports successful!")
print(f"Encoder model: {ENCODER_MODEL_NAME}")

## 2. Load Data

In [ ]:
# Load borderline cases from Phase 8
borderline_df = load_borderline_dataset()
cleaned_df = load_cleaned_dataset()

print(f"\nBorderline cases: {len(borderline_df)}")
print(f"Cleaned data: {len(cleaned_df)}")

In [ ]:
# Merge identity fields for similarity computation
df = merge_identity_fields(borderline_df, cleaned_df)
print(f"\nMerged data shape: {df.shape}")

## 3. Examine Text Fields

Let's look at the text fields we'll use for feature creation.

In [ ]:
# Sample text fields
print("=" * 80)
print("SAMPLE TEXT FIELDS")
print("=" * 80)

sample = df.iloc[0]

print(f"\n--- Application Identity ---")
print(f"Name: {sample['claimed_first_name']} {sample['claimed_last_name']}")
print(f"Address: {sample['address_line']}, {sample['city']}, {sample['state']} {sample['zip_code']}")
print(f"Employer: {sample['employer_name']}")

print(f"\n--- Text Evidence ---")
print(f"Verification Note: {sample['verification_note']}")
print(f"\nOCR Document Text: {sample['ocr_document_text']}")
print(f"\nAddress Explanation: {sample['address_explanation_text']}")
print(f"\nEmployment Explanation: {sample['employment_explanation_text']}")

In [ ]:
# Show how we build reference texts
print("=" * 80)
print("REFERENCE TEXT CONSTRUCTION")
print("=" * 80)

print(f"\n--- Application Identity Text ---")
identity_text = build_application_identity_text(sample)
print(identity_text)

print(f"\n--- Address Reference Text ---")
address_text = build_address_reference_text(sample)
print(address_text)

print(f"\n--- Employer Reference Text ---")
employer_text = build_employer_reference_text(sample)
print(employer_text)

## 4. Load Encoder Model

We use `all-MiniLM-L6-v2`, a lightweight sentence transformer that:
- Is only 80MB
- Runs fast on CPU
- Produces high-quality 384-dimensional embeddings

In [ ]:
# Load the encoder model
model = load_encoder_model()

In [ ]:
# Test the model with a simple example
test_sentences = [
    "Name John Smith Address 123 Main St",
    "John Smith lives at 123 Main Street",
    "Unable to verify identity documents"
]

embeddings = model.encode(test_sentences)
print(f"Embedding shape: {embeddings.shape}")

# Compute similarities
from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(embeddings)

print("\nSimilarity Matrix:")
print(f"  Sentence 1 vs 2 (similar): {sim_matrix[0,1]:.3f}")
print(f"  Sentence 1 vs 3 (different): {sim_matrix[0,2]:.3f}")

## 5. Create Similarity Features

We compute semantic similarity between:
1. **Application identity** vs **OCR document text** - Do they match?
2. **Employer reference** vs **Employment explanation** - Is it consistent?
3. **Address reference** vs **Address explanation** - Is it consistent?

In [ ]:
# Create similarity features
df_with_sim = create_similarity_features(df, model)

In [ ]:
# Examine similarity feature distributions
print("=== Similarity Feature Statistics ===")
sim_cols = ['application_ocr_similarity', 'employment_consistency_score', 'address_consistency_score']
print(df_with_sim[sim_cols].describe())

In [ ]:
# Compare by fraud label
print("\n=== Similarity Features by Fraud Label ===")
for col in sim_cols:
    means = df_with_sim.groupby('fraud_label')[col].mean()
    print(f"\n{col}:")
    print(f"  Legitimate: {means[0]:.4f}")
    print(f"  Fraud: {means[1]:.4f}")
    print(f"  Difference: {means[1] - means[0]:.4f}")

## 6. Create Text Length Features

In [ ]:
# Create text length features
df_with_len = create_text_length_features(df_with_sim)

In [ ]:
# Examine length feature distributions
print("=== Text Length Feature Statistics ===")
len_cols = ['verification_note_length', 'ocr_text_length', 'address_explanation_length', 'employment_explanation_length']
print(df_with_len[len_cols].describe())

## 7. Create Keyword Features

We look for suspicious keywords that might indicate fraud.

In [ ]:
# Show the suspicious keywords we're looking for
print("=== Suspicious Keywords ===")
for i, kw in enumerate(SUSPICIOUS_KEYWORDS, 1):
    print(f"  {i:2d}. {kw}")

In [ ]:
# Create keyword features
df_with_kw = create_keyword_features(df_with_len)

In [ ]:
# Examine keyword feature distributions
print("=== Keyword Feature Statistics ===")
kw_cols = ['suspicious_keyword_count_verification', 'suspicious_keyword_count_ocr', 
           'suspicious_keyword_count_total', 'note_has_high_risk_keyword_flag', 'ocr_has_high_risk_keyword_flag']
print(df_with_kw[kw_cols].describe())

In [ ]:
# Compare keyword features by fraud label
print("\n=== Keyword Features by Fraud Label ===")
for col in kw_cols:
    means = df_with_kw.groupby('fraud_label')[col].mean()
    print(f"\n{col}:")
    print(f"  Legitimate: {means[0]:.4f}")
    print(f"  Fraud: {means[1]:.4f}")

## 8. Create Complete Feature Table

In [ ]:
# Create the complete feature table
features_df = create_text_feature_table(borderline_df, cleaned_df, model)

In [ ]:
# Preview the feature table
print(f"Feature table shape: {features_df.shape}")
print(f"\nColumns:")
for col in features_df.columns:
    print(f"  - {col}")

In [ ]:
# Show sample rows (excluding text columns for readability)
display_cols = [
    'application_id', 'fraud_label', 'fraud_type', 'best_model_score',
    'application_ocr_similarity', 'employment_consistency_score', 'address_consistency_score',
    'suspicious_keyword_count_total', 'note_has_high_risk_keyword_flag'
]
features_df[display_cols].head(10)

## 9. Feature Analysis by Fraud Label

In [ ]:
# Summarize features by fraud label
summary = summarize_features_by_label(features_df)
print("=" * 80)
print("FEATURE MEANS BY FRAUD LABEL")
print("=" * 80)
print(summary.round(4).to_string())

In [ ]:
# Visualize key features by fraud label
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Application-OCR Similarity
ax = axes[0, 0]
for label, color in [(0, 'green'), (1, 'red')]:
    data = features_df[features_df['fraud_label'] == label]['application_ocr_similarity']
    ax.hist(data, bins=15, alpha=0.6, color=color, label=f"{'Legit' if label==0 else 'Fraud'}")
ax.set_xlabel('Application-OCR Similarity')
ax.set_ylabel('Count')
ax.set_title('Application-OCR Similarity by Label')
ax.legend()

# 2. Employment Consistency
ax = axes[0, 1]
for label, color in [(0, 'green'), (1, 'red')]:
    data = features_df[features_df['fraud_label'] == label]['employment_consistency_score']
    ax.hist(data, bins=15, alpha=0.6, color=color, label=f"{'Legit' if label==0 else 'Fraud'}")
ax.set_xlabel('Employment Consistency Score')
ax.set_ylabel('Count')
ax.set_title('Employment Consistency by Label')
ax.legend()

# 3. Suspicious Keyword Count
ax = axes[1, 0]
for label, color in [(0, 'green'), (1, 'red')]:
    data = features_df[features_df['fraud_label'] == label]['suspicious_keyword_count_total']
    ax.hist(data, bins=range(0, int(data.max())+2), alpha=0.6, color=color, label=f"{'Legit' if label==0 else 'Fraud'}")
ax.set_xlabel('Total Suspicious Keyword Count')
ax.set_ylabel('Count')
ax.set_title('Suspicious Keywords by Label')
ax.legend()

# 4. High Risk Keyword Flag Rate
ax = axes[1, 1]
flag_rates = features_df.groupby('fraud_label')['note_has_high_risk_keyword_flag'].mean()
ax.bar(['Legitimate', 'Fraud'], flag_rates.values, color=['green', 'red'], alpha=0.7)
ax.set_ylabel('Rate')
ax.set_title('High Risk Keyword Flag Rate')
for i, v in enumerate(flag_rates.values):
    ax.text(i, v + 0.02, f'{v:.1%}', ha='center')

plt.tight_layout()
plt.show()

## 10. Save Outputs

In [ ]:
# Save the feature table
save_outputs(features_df)

## 11. Summary

In [ ]:
print("=" * 60)
print("PHASE 9 COMPLETE: ENCODER FEATURES SUMMARY")
print("=" * 60)

print(f"\n--- Model Used ---")
print(f"Encoder: {ENCODER_MODEL_NAME}")
print(f"Embedding dimension: 384")

print(f"\n--- Features Created ---")
print("Similarity features (3):")
print("  - application_ocr_similarity")
print("  - employment_consistency_score")
print("  - address_consistency_score")

print("\nLength features (4):")
print("  - verification_note_length")
print("  - ocr_text_length")
print("  - address_explanation_length")
print("  - employment_explanation_length")

print("\nKeyword features (5):")
print("  - suspicious_keyword_count_verification")
print("  - suspicious_keyword_count_ocr")
print("  - suspicious_keyword_count_total")
print("  - note_has_high_risk_keyword_flag")
print("  - ocr_has_high_risk_keyword_flag")

print(f"\n--- Output ---")
print(f"Rows: {len(features_df)}")
print(f"Columns: {len(features_df.columns)}")
print(f"File: data/processed/text_encoder_features.parquet")

print(f"\n--- Key Findings ---")
summary = summarize_features_by_label(features_df)
print(f"Application-OCR similarity: Legit={summary.loc['application_ocr_similarity', 'Legitimate (0)']:.3f}, Fraud={summary.loc['application_ocr_similarity', 'Fraud (1)']:.3f}")
print(f"Suspicious keyword rate: Legit={summary.loc['note_has_high_risk_keyword_flag', 'Legitimate (0)']:.1%}, Fraud={summary.loc['note_has_high_risk_keyword_flag', 'Fraud (1)']:.1%}")

### Interpretation

**Key observations:**

1. **Similarity features show expected patterns:**
   - Fraud cases tend to have lower similarity between claimed identity and OCR text
   - This makes sense - fraudulent documents may not match the application

2. **Keyword features are discriminative:**
   - Fraud cases have more suspicious keywords in verification notes
   - This captures reviewer observations about inconsistencies

3. **Ready for Stage 2 combined model:**
   - These text features can be combined with Stage 1 structured scores
   - The goal is to improve predictions on borderline cases